# C0 + C1: overnight training only

**Suraj — upload this one notebook to `/scratch/sshind38/capstone_c0_c1_work/`.**
Open it in Sol Jupyter. Run cells 1, 2 and 3 below. Wait for **Slurm job IDs**;
after submission, you can close the laptop and let Sol work.

| Model | Original parent seed | Remaining C0 continuation seeds | C1 continuation seeds |
|---|---|---|---|
| 0.5B | 17 | 137, 2026; completed seed 42 is reused | 42, 137, 2026 |
| 0.5B | 123 | 42, 137, 2026 | 42, 137, 2026 |
| 1.5B | 17 | 42, 137, 2026 | 42, 137, 2026 |
| 1.5B | 123 | 42, 137, 2026 | 42, 137, 2026 |

**11 new C0 + 12 new C1 = 23 new training runs. No B training. No benchmark evaluation.**
C1 includes parent generations on training candidates to build its selection;
these are necessary preparation, separate from GSM/MATH benchmark evaluation.

The default is six serial allocations of at most four hours each, one A100 at
a time. This is a **24-hour total cap**, not a runtime estimate. The runner stops
early and cancels unneeded later jobs. Queue time is additional; completion in
one night is not guaranteed. It uses the existing Sol environments and model caches.


## 1. Configuration

The paths below match your completed pilot. Keep the existing B runtime,
screened preview, four parent checkpoints and C0 pilot in place.


In [ ]:
from pathlib import Path
WORK = Path('/scratch/sshind38/capstone_c0_c1_work')
B_ROOT = Path('/scratch/sshind38/capstone_approach_b')
PACKAGE_NAME = 'C0_C1_Overnight_Training_20261110'
TRAIN_PYTHON = B_ROOT / 'venv_b_training_py312/bin/python'
ALLOCATIONS = 6  # Four-hour caps, serial; unused later jobs are cancelled.
ACCOUNT = 'class_fse57081354fall2026'
PARTITION = 'public'
QOS = 'class'
print('New training: C0 = 11, C1 = 12. Benchmark evaluation = OFF.')
print('Work directory:', WORK)


## 2. Install the embedded runner

No second download or ZIP upload is required. This cell verifies the embedded
package and preserves any different existing files. It does not train or submit jobs.


In [ ]:
import base64, hashlib, io, json, zipfile
_EMBEDDED_ZIP_SHA256 = 'bb49d14d52e4dd2d97cc9b4b8744f5896fdfc43946267f0d9ff59c15a25f3100'
_EMBEDDED_PACKAGE = '...'
_content = base64.b64decode(_EMBEDDED_PACKAGE, validate=True)
if hashlib.sha256(_content).hexdigest() != _EMBEDDED_ZIP_SHA256:
    raise ValueError('Embedded package is damaged; download the notebook again.')
if not WORK.is_dir():
    raise FileNotFoundError(f'Open this notebook on Sol. Work directory missing: {WORK}')
if not TRAIN_PYTHON.is_file():
    raise FileNotFoundError(f'Existing B training Python is missing: {TRAIN_PYTHON}')
_destination = WORK / PACKAGE_NAME
with zipfile.ZipFile(io.BytesIO(_content)) as _archive:
    _bad = _archive.testzip()
    if _bad:
        raise ValueError(f'Damaged embedded member: {_bad}')
    _pins = json.loads(_archive.read('PACKAGE_PINS.json'))
    _expected_names = set(_pins) | {'PACKAGE_PINS.json'}
    if set(_archive.namelist()) != _expected_names:
        raise ValueError('Unexpected embedded package members')
    for _member in _archive.infolist():
        _target = (_destination / _member.filename).resolve()
        if not _target.is_relative_to(_destination.resolve()):
            raise ValueError('Unsafe embedded package path')
        _data = _archive.read(_member)
        if _member.filename in _pins and hashlib.sha256(_data).hexdigest() != _pins[_member.filename]:
            raise ValueError(f'Embedded member hash mismatch: {_member.filename}')
        if _target.exists():
            if _target.read_bytes() != _data:
                raise FileExistsError(f'Different existing file preserved: {_target}')
        else:
            _target.parent.mkdir(parents=True, exist_ok=True)
            with _target.open('xb') as _stream:
                _stream.write(_data)
print('Installed and hash-verified:', _destination)
print('No training has started yet. Run cell 3 to check prerequisites and submit.')


## 3. Launch and wait for job IDs

Run this cell once. It checks all four parents, existing pilot, source hashes,
offline model caches and the existing solver/training environments before
submitting. If a check fails, no new training starts.

**Once it prints `submitted_training_not_yet_verified` and the job IDs, you can
close Sol/Jupyter and your laptop.** A pending job means it is queued.
Rerunning this cell reuses an active submission instead of duplicating jobs.


In [ ]:
import subprocess
_runner = WORK / PACKAGE_NAME / 'nightly.py'
_command = [str(TRAIN_PYTHON), '-u', '-B', str(_runner), 'submit',
            '--work', str(WORK), '--b-root', str(B_ROOT), '--allocations', str(ALLOCATIONS),
            '--account', ACCOUNT, '--partition', PARTITION, '--qos', QOS]
subprocess.run(_command, cwd=WORK, check=True)


## 4. Check status later

This cell also works after restarting the notebook kernel. Completion means
**C0 12/12 and C1 12/12**, including the existing C0 pilot. Look for the
`TRAINING_COMPLETE.json` marker, not just a Slurm `COMPLETED` state.


In [ ]:
from pathlib import Path
import subprocess
_work = Path('/scratch/sshind38/capstone_c0_c1_work')
_broot = Path('/scratch/sshind38/capstone_approach_b')
_runner = _work / 'C0_C1_Overnight_Training_20261110/nightly.py'
subprocess.run([str(_broot/'venv_b_training_py312/bin/python'), '-u', '-B', str(_runner),
                'status', '--work', str(_work), '--b-root', str(_broot)], cwd=_work, check=True)


## 5. Show recent job output


In [ ]:
from pathlib import Path
import json
_location = Path('/scratch/sshind38/capstone_c0_c1_work/C0_C1_Overnight_Training_20261110/RUN_LOCATION.json')
if not _location.is_file():
    print('No submitted run is recorded yet. Run the launch cell first.')
else:
    _info = json.loads(_location.read_text())
    print('Control directory:', _info['control_directory'])
    for _job in _info['jobs']:
        print('\nJob', _job['job_id'])
        for _kind in ('stdout', 'stderr'):
            _path = Path(_job[_kind])
            print(_kind + ':', _path)
            if _path.exists():
                print('\n'.join(_path.read_text(errors='replace').splitlines()[-18:]))
            else:
                print('Not created yet; this allocation may still be queued.')


## What will be saved

The launch cell prints the exact output directory. Inside it:

- `STATUS.json`: actual counts, failures and progress.
- `TRAINING_RUNS.csv`: all 24 model/method/seed combinations and adapter locations.
- `TRAINING_COMPLETE.json`: created only after all requested training passes.
- `C0_C1_training_receipts.zip`: small evidence package you can download and share.
- `profiles/`: frozen data, selection manifests, cached scores, candidate generations,
  smoke checks and final trained adapters.
- `stage_logs/` and `slurm_logs/`: exact stage and allocation output.

**Recovery:** compatible completed stages are checked and reused. Interrupted
candidate generation resumes; partial model training restarts from its original
parent in a new preserved attempt. No parent optimizer state is inherited.
If the allocation chain ends before completion, rerun the launch cell for another
bounded chain. Training attempts are limited to three.

**C1 gate:** confirmed mistakes must be sufficiently common and C1 must differ
meaningfully from C0. Failed checks block the affected profile and are reported;
other valid profiles continue. No gate is silently relaxed.

This package has 22 local CPU checks, including a full controller simulation
across an interrupted allocation. **Real GPU training and Sol scheduling have
not been executed here.** The real runs verify adapter changes, reload, original
parent identity, one-pass token consumption and fresh optimizer initialization.
Matching screened B controls beyond your existing pilot and all benchmark
evaluations remain separate. Inherited exposure/screening limitations remain disclosed.
